In [1]:
# (Optional) small environment sanity check
import os, glob

print("cwd:", os.getcwd())
print(
    "Top-level:",
    sorted(os.listdir("/kaggle")) if os.path.exists("/kaggle") else "no /kaggle",
)



cwd: /kaggle/working
Top-level: ['data', 'input', 'working']


In [2]:
import os

# Keep this cell, but make it robust to the provided environment.
# Kaggle typically uses /kaggle/input; your dump also shows /kaggle/data.
for p in ["../input", "/kaggle/input", "/kaggle/data", "/kaggle/working"]:
    if os.path.exists(p):
        print(p, "->", len(os.listdir(p)), "items")
    else:
        print(p, "-> does not exist")



../input -> 8 items
/kaggle/input -> 8 items
/kaggle/data -> 8 items
/kaggle/working -> 2 items


In [3]:
import numpy as np, pandas as pd

# Original code referenced external submission files that aren't available.
# Minimal fix: create four "base submissions" locally (same semantics: probabilities per label),
# then keep the original averaging ensemble unchanged.

# Keep variable names to preserve downstream logic structure.
f_lstm_glove_pl = "base_pred_1.csv"
f_nbsvm = "base_pred_2.csv"
f_lstm_fast_pl = "base_pred_3.csv"
f_bi_lstm_dual = "base_pred_4.csv"

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Resolve data paths (prefer /kaggle/data based on your listing, fallback to /kaggle/input)
base_dir_candidates = [
    "/kaggle/data/jigsaw-toxic-comment-classification-challenge",
    "/kaggle/input/jigsaw-toxic-comment-classification-challenge",
    "/kaggle/data",
    "/kaggle/input",
]
DATA_DIR = None
for c in base_dir_candidates:
    if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(
        os.path.join(c, "test.csv")
    ):
        DATA_DIR = c
        break
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate train.csv/test.csv in expected Kaggle paths."
    )

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

print("Using DATA_DIR:", DATA_DIR)
print(
    "train exists:",
    os.path.exists(train_path),
    "test exists:",
    os.path.exists(test_path),
    "sample exists:",
    os.path.exists(sample_sub_path),
)



Using DATA_DIR: /kaggle/data/jigsaw-toxic-comment-classification-challenge
train exists: True test exists: True sample exists: True


In [4]:
# Train 4 lightweight text models to create the 4 base prediction CSVs (replacing missing ../input submissions).
# This is the smallest change that makes the original averaging ensemble runnable end-to-end in this environment.
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

# Basic null handling
train_text = train_df["comment_text"].fillna("")
test_text = test_df["comment_text"].fillna("")

X_train_text = train_text
y_train = train_df[label_cols].astype(np.int8)

# Build 4 pipelines differing only by random_state / C to mimic "different models" for an average ensemble.
# Keep it simple and deterministic; no approximations/sampling.
configs = [
    dict(random_state=1, C=4.0, ngram_range=(1, 2), min_df=3, max_features=200000),
    dict(random_state=2, C=3.0, ngram_range=(1, 2), min_df=2, max_features=220000),
    dict(random_state=3, C=5.0, ngram_range=(1, 2), min_df=4, max_features=180000),
    dict(random_state=4, C=4.5, ngram_range=(1, 2), min_df=3, max_features=250000),
]

out_files = [f_lstm_glove_pl, f_nbsvm, f_lstm_fast_pl, f_bi_lstm_dual]


def train_and_predict(cfg, out_path):
    # Shared TF-IDF to keep core approach stable; classifier per label (one-vs-rest via separate fits).
    tfidf = TfidfVectorizer(
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\w{1,}",
        ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"],
        max_features=cfg["max_features"],
        sublinear_tf=True,
    )
    Xtr = tfidf.fit_transform(X_train_text)
    Xte = tfidf.transform(test_text)

    pred = pd.DataFrame({"id": test_df["id"].values})
    # Logistic regression per label (common strong baseline for this competition)
    for col in label_cols:
        clf = LogisticRegression(
            solver="liblinear",
            C=cfg["C"],
            random_state=cfg["random_state"],
            max_iter=1000,
        )
        clf.fit(Xtr, y_train[col].values)
        p = clf.predict_proba(Xte)[:, 1]
        # Ensure strictly within [0,1]
        pred[col] = np.clip(p, 0.0, 1.0)

    # Match sample submission column order exactly
    pred = pred[sample_sub.columns.tolist()]
    pred.to_csv(out_path, index=False)
    return pred


# Generate base predictions only if missing (saves time on reruns)
for cfg, out_path in zip(configs, out_files):
    if not os.path.exists(out_path):
        print(
            "Training to create:",
            out_path,
            "with cfg:",
            {
                k: cfg[k]
                for k in ["random_state", "C", "ngram_range", "min_df", "max_features"]
            },
        )
        _ = train_and_predict(cfg, out_path)
    else:
        print("Found existing:", out_path)



Training to create: base_pred_1.csv with cfg: {'random_state': 1, 'C': 4.0, 'ngram_range': (1, 2), 'min_df': 3, 'max_features': 200000}
Training to create: base_pred_2.csv with cfg: {'random_state': 2, 'C': 3.0, 'ngram_range': (1, 2), 'min_df': 2, 'max_features': 220000}
Training to create: base_pred_3.csv with cfg: {'random_state': 3, 'C': 5.0, 'ngram_range': (1, 2), 'min_df': 4, 'max_features': 180000}
Training to create: base_pred_4.csv with cfg: {'random_state': 4, 'C': 4.5, 'ngram_range': (1, 2), 'min_df': 3, 'max_features': 250000}


In [5]:
# Load the 4 base predictions (same as original intent: read CSV submissions)
p_lstm_glove_pl = pd.read_csv(f_lstm_glove_pl)
p_nbsvm = pd.read_csv(f_nbsvm)
p_lstm_fast_pl = pd.read_csv(f_lstm_fast_pl)
p_bi_lstm_dual = pd.read_csv(f_bi_lstm_dual)


# Safety: ensure consistent ordering by id before ensembling
def sort_by_id(df):
    if "id" not in df.columns:
        raise ValueError("Prediction file missing 'id' column")
    return df.sort_values("id").reset_index(drop=True)


p_lstm_glove_pl = sort_by_id(p_lstm_glove_pl)
p_nbsvm = sort_by_id(p_nbsvm)
p_lstm_fast_pl = sort_by_id(p_lstm_fast_pl)
p_bi_lstm_dual = sort_by_id(p_bi_lstm_dual)

# Verify alignment
ids = p_lstm_glove_pl["id"].values
for name, df in [
    ("nbsvm", p_nbsvm),
    ("fast", p_lstm_fast_pl),
    ("dual", p_bi_lstm_dual),
]:
    if not np.array_equal(ids, df["id"].values):
        raise ValueError(f"ID mismatch between lstm_glove and {name} predictions")



In [6]:
# Original averaging ensemble (preserved)
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
p_res = p_lstm_glove_pl.copy()
p_res[label_cols] = (
    p_nbsvm[label_cols]
    + p_lstm_glove_pl[label_cols]
    + p_lstm_fast_pl[label_cols]
    + p_bi_lstm_dual[label_cols]
) / 4.0

# Keep output columns exactly as required
p_res = p_res[["id"] + label_cols]

# Final numeric safety
p_res[label_cols] = p_res[label_cols].astype(np.float32).clip(0.0, 1.0)



In [7]:
# Quick validation against sample submission format
sample_cols = list(sample_sub.columns)
if list(p_res.columns) != sample_cols:
    raise ValueError(
        f"Submission columns mismatch.\nExpected: {sample_cols}\nGot: {list(p_res.columns)}"
    )

if len(p_res) != len(sample_sub):
    # In this competition, test rows should equal sample submission rows
    print(
        "Warning: row count differs from sample_submission:",
        len(p_res),
        "vs",
        len(sample_sub),
    )

print(p_res.head())
print("Submission shape:", p_res.shape)



                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.999484      0.127565  0.997604  0.040207  0.925090   
1  0000247867823ef7  0.002553      0.000935  0.001137  0.000303  0.003002   
2  00013b17ad220c46  0.012228      0.001814  0.006765  0.000682  0.006120   
3  00017563c3f7919a  0.003003      0.001462  0.002372  0.000819  0.001944   
4  00017695ad8997eb  0.012037      0.003298  0.004994  0.001237  0.003753   

   identity_hate  
0       0.299252  
1       0.001282  
2       0.001809  
3       0.000838  
4       0.000926  
Submission shape: (153164, 7)


In [8]:
# Write the required Kaggle submission file
p_res.to_csv("submission.csv", index=False)
print("Wrote submission.csv")

Wrote submission.csv
